# SAM + I-JEPA StyleGAN2 Training - Debug Setup

## Recent Improvements (Feb 3, 2026)

### 1. I-JEPA Fusion Improvements
- **`fusion_depth`**: 4 → **6** (more semantic layers)
- **`sem_mixing_prob`**: 0.70 → **0.0** (no random truncation = consistent conditioning)
- **`fusion_alpha`**: 0.2 → **0.3** with **tanh bounding** (stability + strength)
- **`ijepa_input_channel`**: 1 → **3** (RGB images, not grayscale)

### 2. Debug Mode
- **DataLoader workers disabled** when `STYLEGAN_DEBUG=1` (prevents debugger hang)
- Split DataLoader construction for clearer debugging

### 3. Parallel SAM Extraction
- Multi-GPU pre-extraction at training start
- File locking for safe concurrent cache writes
- Strided indexing `[rank::world_size]` for work distribution

## Usage
Run Cell 2 to start training with improved fusion parameters.

In [1]:
import os
os.environ['STYLEGAN_DEBUG'] = '1'

In [1]:
import importlib
import sys
import os

# Enable debug mode for DataLoader
os.environ['STYLEGAN_DEBUG'] = '1'

repo_root = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2"
# Use a small subset for debugging - create this manually with 10-20 images
data_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagenet_debug_subset.zip"
out_base = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2/outputs/debug_sam"

# SAM checkpoint for on-the-fly extraction
sam_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/segProto/checkpoints/sam_vit_b_01ec64.pth"

# IJEPA checkpoint (REQUIRED for training)
ijepa_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/pretrained_enc_ckpts/ijepa/IN1K-vit.h.14-300e.pth.tar"

# Debug settings
SAM_PROB = 1.0  # Always use SAM masks for debugging
MAX_IMAGES = 20  # Limit dataset size

outdir = f"{out_base}/sam_debug_run/training-runs"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import train
importlib.reload(train)

# SAM debugging configuration
sys.argv = [
    "train.py",
    "--outdir", outdir,
    "--data", data_path,
    "--gpus", "1",
    "--cond", "0",  # No class conditioning, only SAM
    "--batch", "4",  # Small batch for debugging
    "--kimg", "10",  # Just 10k images total for quick test
    
    # I-JEPA (REQUIRED - needed for encoder initialization)
    "--ijepa_checkpoint", ijepa_checkpoint,
    "--ijepa_lambda", "0.0",  # Set to 0 if you don't want IJEPA loss, just SAM
    "--extra_dim", "384",  # Match your IJEPA embedding dimension
    
    # SAM on-the-fly extraction
    "--sam-enabled", "True",  # Enable SAM extraction
    "--sam-checkpoint", sam_checkpoint,
    "--sam-cache-dir", f"{out_base}/sam_cache",  # Required cache directory
    "--sam-model-type", "vit_b",
    "--sam-prob", str(SAM_PROB),  # 1.0 = always use SAM
    "--sam-max-masks", "250",  # Max masks per image
    
    # Dataset limiting
    "--subset", str(MAX_IMAGES),
    
    # Fast debugging
    "--snap", "1",  # Save snapshot every 1 tick
    "--metrics", "none",  # Skip metric computation
    
    "--resume", "noresume"

]

print("="*60)
print("SAM DEBUG CONFIGURATION (IMPROVED)")
print("="*60)
print(f"Dataset: {data_path}")
print(f"Output: {outdir}")
print(f"SAM Probability: {SAM_PROB} (always ON)")
print(f"Max Images: {MAX_IMAGES}")
print(f"On-the-fly SAM: YES (no NPZ needed)")
print(f"SAM Cache Dir: {out_base}/sam_cache")
print(f"IJEPA Checkpoint: {ijepa_checkpoint}")
print()
print("I-JEPA FUSION IMPROVEMENTS:")
print(f"  • fusion_depth: 6 (was 4)")
print(f"  • sem_mixing_prob: 0.0 (was 0.70) - consistent fusion")
print(f"  • fusion_alpha: 0.3 (was 0.2) - bounded by tanh")
print(f"  • ijepa_input_channel: 3 (RGB, not grayscale)")
print("="*60)
print()

train.main(standalone_mode=False)

SAM DEBUG CONFIGURATION
Dataset: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagenet_debug_subset.zip
Output: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2/outputs/debug_sam/sam_debug_run/training-runs
SAM Probability: 1.0 (always ON)
Max Images: 20
On-the-fly SAM: YES (no NPZ needed)
SAM Cache Dir: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2/outputs/debug_sam/sam_cache
IJEPA Checkpoint: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/pretrained_enc_ckpts/ijepa/IN1K-vit.h.14-300e.pth.tar

⚠️  IMPORTANT: Update the ijepa_checkpoint path before running!

[3, 256, 256]

Training options:
{
  "num_gpus": 1,
  "image_snapshot_ticks": 1,
  "network_snapshot_ticks": 1,
  "metrics": [],
  "random_seed": 0,
  "training_set_kwargs": {
    "class_name": "training.dataset.ImageFolderDataset",
    "path": "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagen

2026-02-03 16:16:48.728734: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770153408.755553 3365273 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770153408.763841 3365273 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770153408.784007 3365273 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770153408.784045 3365273 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770153408.784048 3365273 computation_placer.cc:177] computation placer alr

Training for 10 kimg...

tick 0     kimg 0.0      time 40m 57s      sec/tick 2330.2  sec/kimg 582541.70 maintenance 126.7  cpumem 3.62   gpumem 7.06   augment 0.000



Abort: 

In [3]:
import torch
print(torch.__version__)

2.7.0+cu126
